# Delta Lake, from first principles

This notebook is a **tutorial**, not a design document. Everything in it runs
against a scratch bucket (`tutorial-veloz`) with a toy dataset that has
nothing to do with Veloz's real orders/fulfillment/payments data. Nothing
here is a proposal for how `bronze.orders` should look — that schema and
partitioning decision is yours to make, per `CLAUDE.md`'s operating model,
once you understand the mechanics well enough to make it deliberately.

**Goal:** by the end of this notebook you should be able to explain, from
having actually watched it happen (not just read about it):

1. What a Delta table physically *is* on object storage, and why that answers
   Julián's audit ask (G2) and Marcela/Ana's "don't babysit it" ask (G3).
2. Schema enforcement vs. schema evolution, and why the default is
   enforcement.
3. What partitioning actually changes on disk and in a query plan, and the
   two rules of thumb for choosing a partition column.
4. Time travel: how `DESCRIBE HISTORY` and `VERSION AS OF` work, and why
   `VACUUM` is the one operation that actually deletes something.

**Sources used, not just recalled from memory** (you asked for this
explicitly): the official Delta Lake docs —
[Table batch reads and writes](https://docs.delta.io/latest/delta-batch.html),
[Best practices](https://docs.delta.io/latest/best-practices.html), and
[Utility commands](https://docs.delta.io/latest/delta-utility.html).
Quotes below are pulled from those pages.


## 0. What Delta Lake actually is

Parquet alone is just files. Two systems reading/writing the same Parquet
directory at once can corrupt it, there's no atomic multi-file commit, and
"what did this table look like yesterday" requires you to have kept a copy
yourself.

Delta Lake is a **storage layer**: still Parquet files underneath, plus a
transaction log directory (`_delta_log/`) sitting next to the data files.
Every write — a plain write, an overwrite, a `MERGE`, a `DELETE` — appends
one new JSON commit file to that log describing exactly which data files
became part of the table and which stopped being part of it. The docs put it
plainly: *"the table's transaction log at the location is the source of
truth."* Not a metastore entry, not a convention about file naming — the log.

That log is what buys two things this project actually needs:

- **G2 (Julián's audit ask):** every version the table ever had is
  reconstructable from the log, so "what did yesterday's numbers look like"
  is a query (`VERSION AS OF`), not a hope that someone kept a backup.
- **G3 (resilience):** an `overwrite` or a `MERGE` never deletes a physical
  file — it writes new files and a new commit that says "old files are no
  longer part of the current version." A bad DAG run producing wrong data is
  a bad *version*, and the previous good version is still sitting right
  there to travel back to. Only `VACUUM` physically deletes anything, which
  we'll get to.

We're about to watch this happen for real, not take it on faith.


## 1. Set up a Spark session that can talk to MinIO

This notebook is meant to run in the `jupyter` docker-compose service
(`docker compose up -d jupyter`, then open <http://localhost:8888>), **not**
the host `.venv`. Two reasons, found empirically while building this, not
assumed:

- The host venv is on Python 3.14, outside pyspark 3.5.3's documented support
  range (3.8–3.11). `spark.range(...)` alone works fine there, but creating a
  DataFrame from Python-side data does not — it fails with a `PicklingError` /
  `RecursionError` from `cloudpickle`, a real Python-3.14 incompatibility, not
  a fluke. The `jupyter` service reuses the exact same image as the Airflow
  containers (Python 3.11), so it doesn't hit this.
- Inside that service, MinIO is reachable at the in-Docker address
  `http://minio:9000`, already exposed as `$MINIO_ENDPOINT`.

Credentials: **root** (`$MINIO_ROOT_USER` / `$MINIO_ROOT_PASSWORD`), not the
`veloz-ingest` / `veloz-maintenance` roles the real pipeline uses. Those two
are scoped by IAM policy to exactly `bronze-veloz` / `silver-veloz` /
`gold-veloz` (see `docs/minio-user-guide.md`) — `tutorial-veloz` is
deliberately outside that policy, so root is the only credential that can
reach it. That's the point: nothing in this notebook can touch the
production buckets even by mistake.


In [2]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ROOT_USER = os.environ["MINIO_ROOT_USER"]
MINIO_ROOT_PASSWORD = os.environ["MINIO_ROOT_PASSWORD"]
BUCKET = "tutorial-veloz"

builder = (
    SparkSession.builder.appName("delta-lake-tutorial")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ROOT_USER)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_ROOT_PASSWORD)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)
spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
from importlib.metadata import version as pkg_version
print("delta-spark version:", pkg_version("delta-spark"))


Spark version: 3.5.3
delta-spark version: 3.2.1


## 2. A toy dataset — library checkouts

Deliberately not orders/fulfillment/payments shaped, so there's no
temptation to read this as a Bronze design. A public library's checkout log:
one row per book checked out of one of a few branches.


In [3]:
from datetime import date

checkouts = spark.createDataFrame(
    [
        (1, "downtown",  "B-1001", "M-501", date(2026, 1, 5),  0),
        (2, "downtown",  "B-1002", "M-502", date(2026, 1, 5),  3),
        (3, "eastside",  "B-1003", "M-503", date(2026, 1, 6),  0),
        (4, "eastside",  "B-1001", "M-504", date(2026, 1, 6),  0),
        (5, "westside",  "B-1004", "M-505", date(2026, 1, 7),  1),
        (6, "downtown",  "B-1005", "M-506", date(2026, 1, 7),  0),
        (7, "westside",  "B-1002", "M-507", date(2026, 1, 8),  0),
        (8, "eastside",  "B-1006", "M-508", date(2026, 1, 8),  5),
    ],
    schema="checkout_id INT, branch STRING, book_id STRING, member_id STRING, checkout_date DATE, days_late INT",
)
checkouts.show()
checkouts.printSchema()


+-----------+--------+-------+---------+-------------+---------+
|checkout_id|  branch|book_id|member_id|checkout_date|days_late|
+-----------+--------+-------+---------+-------------+---------+
|          1|downtown| B-1001|    M-501|   2026-01-05|        0|
|          2|downtown| B-1002|    M-502|   2026-01-05|        3|
|          3|eastside| B-1003|    M-503|   2026-01-06|        0|
|          4|eastside| B-1001|    M-504|   2026-01-06|        0|
|          5|westside| B-1004|    M-505|   2026-01-07|        1|
|          6|downtown| B-1005|    M-506|   2026-01-07|        0|
|          7|westside| B-1002|    M-507|   2026-01-08|        0|
|          8|eastside| B-1006|    M-508|   2026-01-08|        5|
+-----------+--------+-------+---------+-------------+---------+

root
 |-- checkout_id: integer (nullable = true)
 |-- branch: string (nullable = true)
 |-- book_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- checkout_date: date (nullable = true)
 |-- days

## 3. Your first Delta table — write it, then look underneath

`.format("delta")` is the only thing that makes this a Delta table instead
of a plain Parquet write. No partitioning yet — that's its own section.


In [4]:
table_path = f"s3a://{BUCKET}/checkouts"

checkouts.write.format("delta").mode("overwrite").save(table_path)
print("Wrote to", table_path)


Wrote to s3a://tutorial-veloz/checkouts


Now look at what actually landed on MinIO — not through Spark, but with `mc`
(the MinIO client), so we're looking at real files, not Spark's idea of the
table. This image bundles the `mc` binary for exactly this.


In [5]:
import subprocess

def mc(*args):
    """Run an mc command and print its output."""
    result = subprocess.run(["mc", *args], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

mc("alias", "set", "veloz", MINIO_ENDPOINT, MINIO_ROOT_USER, MINIO_ROOT_PASSWORD)
mc("ls", "--recursive", f"veloz/{BUCKET}/checkouts")


Added `veloz` successfully.

[2026-08-28 16:58:05 UTC] 5.9KiB STANDARD _delta_log/00000000000000000000.json
[2026-08-28 16:58:05 UTC]     0B STANDARD _delta_log/_commits/
[2026-08-28 16:58:04 UTC]   769B STANDARD part-00000-66d3dede-f4c4-493a-846d-784fed901ac9-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00001-04e135ad-b620-4d8f-95ed-14530f1899cf-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00003-35486348-2612-4bf1-a873-4054ee3e747a-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00005-bba77dc0-ca14-4305-b98d-3a4104ff37a5-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00006-a24432df-a7f9-48b7-a583-e617b45ad9ec-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00008-a7d159c7-fa3f-4227-a9b0-390455553830-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00010-0e01e2d8-2436-4027-96a0-046ca618ffed-c000.snappy.parquet
[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00012-9b7ec9

CompletedProcess(args=['mc', 'ls', '--recursive', 'veloz/tutorial-veloz/checkouts'], returncode=0, stdout='[2026-08-28 16:58:05 UTC] 5.9KiB STANDARD _delta_log/00000000000000000000.json\n[2026-08-28 16:58:05 UTC]     0B STANDARD _delta_log/_commits/\n[2026-08-28 16:58:04 UTC]   769B STANDARD part-00000-66d3dede-f4c4-493a-846d-784fed901ac9-c000.snappy.parquet\n[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00001-04e135ad-b620-4d8f-95ed-14530f1899cf-c000.snappy.parquet\n[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00003-35486348-2612-4bf1-a873-4054ee3e747a-c000.snappy.parquet\n[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00005-bba77dc0-ca14-4305-b98d-3a4104ff37a5-c000.snappy.parquet\n[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00006-a24432df-a7f9-48b7-a583-e617b45ad9ec-c000.snappy.parquet\n[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00008-a7d159c7-fa3f-4227-a9b0-390455553830-c000.snappy.parquet\n[2026-08-28 16:58:04 UTC] 1.7KiB STANDARD part-00010-0e01e2d8-2436-4027-96a0-046ca6

Two kinds of files:

- `part-*.snappy.parquet` — the actual data, exactly what a plain Parquet
  write would have produced.
- `_delta_log/00000000000000000000.json` — the **first commit**. This is the
  entire difference between "a folder of Parquet files" and "a Delta table."

Let's read that commit file directly.


In [10]:
result = mc("cat", f"veloz/{BUCKET}/checkouts/_delta_log/00000000000000000000.json")
display(result)

{"commitInfo":{"timestamp":1787936285169,"operation":"WRITE","operationParameters":{"mode":"Overwrite","partitionBy":"[]"},"isolationLevel":"Serializable","isBlindAppend":false,"operationMetrics":{"numFiles":"9","numOutputRows":"8","numOutputBytes":"14783"},"engineInfo":"Apache-Spark/3.5.3 Delta-Lake/3.2.1","txnId":"fb62e20c-14de-414f-975e-271b0ff64c67"}}
{"metaData":{"id":"4160d450-aead-4e4c-980e-5031e58094db","format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"checkout_id\",\"type\":\"integer\",\"nullable\":true,\"metadata\":{}},{\"name\":\"branch\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"book_id\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"member_id\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"checkout_date\",\"type\":\"date\",\"nullable\":true,\"metadata\":{}},{\"name\":\"days_late\",\"type\":\"integer\",\"nullable\":true,\"metadata\":{}}]}","partitionCol

CompletedProcess(args=['mc', 'cat', 'veloz/tutorial-veloz/checkouts/_delta_log/00000000000000000000.json'], returncode=0, stdout='{"commitInfo":{"timestamp":1787936285169,"operation":"WRITE","operationParameters":{"mode":"Overwrite","partitionBy":"[]"},"isolationLevel":"Serializable","isBlindAppend":false,"operationMetrics":{"numFiles":"9","numOutputRows":"8","numOutputBytes":"14783"},"engineInfo":"Apache-Spark/3.5.3 Delta-Lake/3.2.1","txnId":"fb62e20c-14de-414f-975e-271b0ff64c67"}}\n{"metaData":{"id":"4160d450-aead-4e4c-980e-5031e58094db","format":{"provider":"parquet","options":{}},"schemaString":"{\\"type\\":\\"struct\\",\\"fields\\":[{\\"name\\":\\"checkout_id\\",\\"type\\":\\"integer\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"branch\\",\\"type\\":\\"string\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"book_id\\",\\"type\\":\\"string\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"member_id\\",\\"type\\":\\"string\\",\\"nullable\\":true,\\"metadata

Each line is a separate JSON action. You should see, among others:

- a `commitInfo` action — operation (`WRITE`), mode (`Overwrite`), timestamp.
- a `metaData` action — the table's schema, in Delta's own JSON schema
  representation, captured *at commit time*.
- one `add` action per Parquet file that became part of this version, each
  with its path, size, and stats.

This is why the docs call the log "the source of truth": everything needed
to know what the table looked like at this version is right here, not
inferred from listing the directory.


## 4. Reading it back, and `DESCRIBE HISTORY`


In [11]:
from delta.tables import DeltaTable

df = spark.read.format("delta").load(table_path)
df.orderBy("checkout_id").show()

dt = DeltaTable.forPath(spark, table_path)
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)


+-----------+--------+-------+---------+-------------+---------+
|checkout_id|  branch|book_id|member_id|checkout_date|days_late|
+-----------+--------+-------+---------+-------------+---------+
|          1|downtown| B-1001|    M-501|   2026-01-05|        0|
|          2|downtown| B-1002|    M-502|   2026-01-05|        3|
|          3|eastside| B-1003|    M-503|   2026-01-06|        0|
|          4|eastside| B-1001|    M-504|   2026-01-06|        0|
|          5|westside| B-1004|    M-505|   2026-01-07|        1|
|          6|downtown| B-1005|    M-506|   2026-01-07|        0|
|          7|westside| B-1002|    M-507|   2026-01-08|        0|
|          8|eastside| B-1006|    M-508|   2026-01-08|        5|
+-----------+--------+-------+---------+-------------+---------+

+-------+-------------------+---------+--------------------------------------+
|version|timestamp          |operation|operationParameters                   |
+-------+-------------------+---------+----------------------

## 5. Schema enforcement vs. schema evolution

Default behavior is **enforcement**. From the docs: *"All DataFrame columns
must exist in the target table. If there are columns in the DataFrame not
present in the table, an exception is raised."*

Let's actually trigger that, rather than take it on faith.


In [12]:
from datetime import date as _date

new_branch_checkouts = spark.createDataFrame(
    [(9, "northside", "B-1007", "M-509", _date(2026, 1, 9), 0, "self-checkout")],
    schema="checkout_id INT, branch STRING, book_id STRING, member_id STRING, "
           "checkout_date DATE, days_late INT, channel STRING",
)

try:
    new_branch_checkouts.write.format("delta").mode("append").save(table_path)
    print("Write succeeded (unexpected)")
except Exception as e:
    print(f"Write rejected, as expected: {type(e).__name__}")
    print(str(e)[:400])


Write rejected, as expected: AnalysisException
[_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 4160d450-aead-4e4c-980e-5031e58094db).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific t


That `channel` column doesn't exist in the table yet, so the write was
rejected before touching storage — this is schema enforcement doing exactly
what it's for. Now the opt-in path: `mergeSchema`. Per the docs, this is
meant to be turned on **per write**, not session-wide (*"Enabling schema
evolution session-wide is not recommended because it can lead to unintended
schema changes"*) — an accidental typo'd column name would otherwise silently
become a permanent new column instead of an error.


In [13]:
(
    new_branch_checkouts.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(table_path)
)

df2 = spark.read.format("delta").load(table_path)
df2.orderBy("checkout_id").show()
df2.printSchema()


+-----------+---------+-------+---------+-------------+---------+-------------+
|checkout_id|   branch|book_id|member_id|checkout_date|days_late|      channel|
+-----------+---------+-------+---------+-------------+---------+-------------+
|          1| downtown| B-1001|    M-501|   2026-01-05|        0|         NULL|
|          2| downtown| B-1002|    M-502|   2026-01-05|        3|         NULL|
|          3| eastside| B-1003|    M-503|   2026-01-06|        0|         NULL|
|          4| eastside| B-1001|    M-504|   2026-01-06|        0|         NULL|
|          5| westside| B-1004|    M-505|   2026-01-07|        1|         NULL|
|          6| downtown| B-1005|    M-506|   2026-01-07|        0|         NULL|
|          7| westside| B-1002|    M-507|   2026-01-08|        0|         NULL|
|          8| eastside| B-1006|    M-508|   2026-01-08|        5|         NULL|
|          9|northside| B-1007|    M-509|   2026-01-09|        0|self-checkout|
+-----------+---------+-------+---------

Notice the earlier 8 rows now show `channel = NULL` — Delta didn't backfill
or guess a value, it just recorded that those rows predate the column. This
distinction (enforce by default, evolve deliberately) matters for
`bronze.orders`: an upstream schema drift should probably fail loudly, not
silently reshape the table underneath you — that's exactly the G3 scenario
CLAUDE.md flags.


## 6. Partitioning — what it changes on disk and in a query plan

Partitioning writes each distinct value of a column into its own
subdirectory (Hive-style: `branch=downtown/`, `branch=eastside/`, ...),
so a query that filters on that column can skip reading the other
directories entirely — "partition pruning."

The official guidance is two rules of thumb, not a vague "partition
whatever you query by":

1. *"If the cardinality of a column will be very high, do not use that
   column for partitioning"* (the docs' example of what **not** to do:
   partitioning by `userId` with millions of distinct values — that produces
   millions of tiny directories, which is worse than no partitioning).
2. *"You can partition by a column if you expect data in that partition to
   be at least 1 GB."* Below that, you mostly get a directory listing
   overhead tax with no read-time benefit. The docs call out `date` as the
   most common real-world partition column, which fits Bronze's daily-extract
   ingestion pattern well — worth keeping in mind for later, not deciding here.

Our toy dataset is far too small for the 1GB guidance to mean anything — this
section is about seeing the *mechanism*, not a performance benchmark.


In [14]:
partitioned_path = f"s3a://{BUCKET}/checkouts_partitioned"

checkouts.write.format("delta").mode("overwrite").partitionBy("branch").save(partitioned_path)

mc("ls", "--recursive", f"veloz/{BUCKET}/checkouts_partitioned")


[2026-08-28 17:31:11 UTC] 5.8KiB STANDARD _delta_log/00000000000000000000.json
[2026-08-28 17:31:11 UTC]     0B STANDARD _delta_log/_commits/
[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=downtown/part-00001-eba73546-3278-4dc6-bf31-4691d6e7e4d1.c000.snappy.parquet
[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=downtown/part-00003-7cfaa91c-6bb8-4947-82bb-cf45aa764c8c.c000.snappy.parquet
[2026-08-28 17:31:11 UTC] 1.4KiB STANDARD branch=downtown/part-00010-3c267cb1-9608-4802-8e44-d93ecec25a56.c000.snappy.parquet
[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=eastside/part-00005-e52099f7-2cad-4996-b30b-8f6071a0838c.c000.snappy.parquet
[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=eastside/part-00006-1664f2d8-50ac-4d03-8618-81c0012051a5.c000.snappy.parquet
[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=eastside/part-00013-51d8def0-22db-4c18-928b-b89a1aece0ab.c000.snappy.parquet
[2026-08-28 17:31:11 UTC] 1.4KiB STANDARD branch=westside/part-00008-601f7546-890c-415c-b07b-920af6b32

CompletedProcess(args=['mc', 'ls', '--recursive', 'veloz/tutorial-veloz/checkouts_partitioned'], returncode=0, stdout='[2026-08-28 17:31:11 UTC] 5.8KiB STANDARD _delta_log/00000000000000000000.json\n[2026-08-28 17:31:11 UTC]     0B STANDARD _delta_log/_commits/\n[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=downtown/part-00001-eba73546-3278-4dc6-bf31-4691d6e7e4d1.c000.snappy.parquet\n[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=downtown/part-00003-7cfaa91c-6bb8-4947-82bb-cf45aa764c8c.c000.snappy.parquet\n[2026-08-28 17:31:11 UTC] 1.4KiB STANDARD branch=downtown/part-00010-3c267cb1-9608-4802-8e44-d93ecec25a56.c000.snappy.parquet\n[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=eastside/part-00005-e52099f7-2cad-4996-b30b-8f6071a0838c.c000.snappy.parquet\n[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=eastside/part-00006-1664f2d8-50ac-4d03-8618-81c0012051a5.c000.snappy.parquet\n[2026-08-28 17:31:11 UTC] 1.5KiB STANDARD branch=eastside/part-00013-51d8def0-22db-4c18-928b-b89a1aece

Compare that listing to the flat one from `checkouts/` above: instead of
Parquet files sitting directly under the table root, they're nested under
`branch=downtown/`, `branch=eastside/`, `branch=westside/`. `_delta_log/`
stays at the table root either way — partitioning is purely a data-layout
decision, the log format doesn't change.


### Does it actually change the query plan?


In [15]:
print("--- unpartitioned table, filtered on branch ---")
spark.read.format("delta").load(table_path).filter(col("branch") == "downtown").explain()

print("--- partitioned table, filtered on branch ---")
spark.read.format("delta").load(partitioned_path).filter(col("branch") == "downtown").explain()


--- unpartitioned table, filtered on branch ---
== Physical Plan ==
*(1) Filter (isnotnull(branch#3551) AND (branch#3551 = downtown))
+- *(1) ColumnarToRow
   +- FileScan parquet [checkout_id#3550,branch#3551,book_id#3552,member_id#3553,checkout_date#3554,days_late#3555,channel#3556] Batched: true, DataFilters: [isnotnull(branch#3551), (branch#3551 = downtown)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[s3a://tutorial-veloz/checkouts], PartitionFilters: [], PushedFilters: [IsNotNull(branch), EqualTo(branch,downtown)], ReadSchema: struct<checkout_id:int,branch:string,book_id:string,member_id:string,checkout_date:date,days_late...


--- partitioned table, filtered on branch ---
== Physical Plan ==
*(1) Project [checkout_id#3755, branch#3756, book_id#3757, member_id#3758, checkout_date#3759, days_late#3760]
+- *(1) ColumnarToRow
   +- FileScan parquet [checkout_id#3755,book_id#3757,member_id#3758,checkout_date#3759,days_late#3760,branch#3756] Batched: true, DataFilters: [

On the partitioned read you should see `PartitionFilters` in the physical
plan alongside (or instead of) `PushedFilters` — Spark knows it can skip
entire directories before ever opening a Parquet file. On the unpartitioned
table, `branch == downtown` can only be a `PushedFilters` predicate applied
row-by-row inside each file it has to open anyway. At our toy scale the time
difference is imperceptible; at millions of rows, skipping whole directories
instead of scanning them is the entire benefit partitioning buys you.


## 7. Time travel

We now have two versions of `checkouts` (`v0` = the original 8 rows, `v1` =
the `mergeSchema` append). Delta lets you read any prior version directly.


In [16]:
dt = DeltaTable.forPath(spark, table_path)
print("Full history:")
dt.history().select("version", "timestamp", "operation").show(truncate=False)

print("Version 0, read back (channel column shouldn't exist yet):")
spark.read.format("delta").option("versionAsOf", 0).load(table_path).printSchema()

print("Latest version:")
spark.read.format("delta").option("versionAsOf", 1).load(table_path).printSchema()


Full history:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-08-28 17:19:07|WRITE    |
|0      |2026-08-28 16:58:05|WRITE    |
+-------+-------------------+---------+

Version 0, read back (channel column shouldn't exist yet):
root
 |-- checkout_id: integer (nullable = true)
 |-- branch: string (nullable = true)
 |-- book_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- checkout_date: date (nullable = true)
 |-- days_late: integer (nullable = true)

Latest version:
root
 |-- checkout_id: integer (nullable = true)
 |-- branch: string (nullable = true)
 |-- book_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- checkout_date: date (nullable = true)
 |-- days_late: integer (nullable = true)
 |-- channel: string (nullable = true)



Equivalent SQL, for reference (not run here since this table isn't
registered in a metastore):

```sql
SELECT * FROM delta.`s3a://tutorial-veloz/checkouts` VERSION AS OF 0
SELECT * FROM delta.`s3a://tutorial-veloz/checkouts` TIMESTAMP AS OF '2026-01-05'
```

**`VACUUM`** is the one operation that actually deletes files — it removes
data files no longer referenced by *any* version newer than the retention
threshold (default **7 days**). The docs are explicit about the tradeoff:
*"The ability to time travel back to a version older than the retention
period is lost after running vacuum."* That's exactly why
`docs/minio-user-guide.md` splits MinIO access into two roles: the ingest
role Airflow uses for scheduled writes physically **cannot** delete
anything, and only a deliberately-invoked maintenance job with the separate
`veloz-maintenance` credentials can run `VACUUM`. We're not running it here —
nothing to reclaim yet, and it's not a command to fire on a whim.


## 8. Recap

What actually happened in this notebook, in order:

1. Wrote a plain DataFrame with `.format("delta")` → got a Parquet directory
   *plus* a `_delta_log/` commit — inspected that commit file directly on
   MinIO via `mc`, not just trusted Spark's description of it.
2. Read the table back, and pulled `DESCRIBE HISTORY`-equivalent metadata
   via `DeltaTable.history()`.
3. Triggered a real schema-enforcement rejection, then opted into evolution
   explicitly with `mergeSchema`, and saw the old rows null-pad rather than
   silently reinterpret.
4. Wrote the same data partitioned by `branch`, saw the Hive-style directory
   layout on MinIO, and confirmed `PartitionFilters` shows up in the
   physical plan for a partitioned read but not an unpartitioned one.
5. Read two different versions of the same table back with `versionAsOf`,
   and saw why `VACUUM` is kept behind a separate, deliberately-invoked
   credential.

**What's next, and whose call it is:** everything above lives in
`tutorial-veloz`, on throwaway data, and stays there — it doesn't touch
`bronze-veloz`. Designing `bronze.orders`' actual schema (explicit vs.
inferred, which ingestion-metadata columns) and its actual partitioning
column (if any) against the real `orders_<date>.csv` extracts is the next
piece of platform-build work, and per `CLAUDE.md` that design call is yours,
not something this notebook decided for you.

**Cited throughout:**
[Table batch reads and writes](https://docs.delta.io/latest/delta-batch.html) ·
[Best practices](https://docs.delta.io/latest/best-practices.html) ·
[Utility commands](https://docs.delta.io/latest/delta-utility.html)


---
Run the cell below when you're done experimenting to release the Spark
session. Skip it if you want to keep poking at `spark`/`checkouts`/`dt`
interactively.


In [ ]:
spark.stop()
